# 04 — Paired patch extraction and WSI reconstruction

Paired extraction reads an H&E reference and an already aligned IHC WSI at
the same physical magnification and target-grid coordinates. It keeps a
pair only when the H&E patch passes the tissue threshold and records every
coordinate in a JSON manifest.

This notebook contains a tiny synthetic example followed by a real-data
template. Install `.[extraction,viz]`; reconstruction also requires the
native libvips runtime.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    '''Find the RocqiPath repository whether Jupyter starts at root or how_to_use.'''
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "rocqipath"
        ).is_dir():
            return candidate
    raise FileNotFoundError(
        "RocqiPath repository not found. Start Jupyter inside the cloned repository."
    )


PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_ROOT = PROJECT_ROOT / "results"

print(f"Project : {PROJECT_ROOT}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")


## Filename rule

The default reference pattern accepts sample IDs containing underscores or
hyphens and common WSI extensions. Override `reference_pattern` only when
your filenames need a stricter rule. A custom pattern must contain a named
`sample_id` group, for example:

```python
r"^(?P<sample_id>Sample_\d{4})_he\.tiff?$"
```


In [ ]:
import os

import numpy as np
from PIL import Image

from rocqipath.extraction import PatchExtractionConfig, run_patch_extraction

demo_root = Path(
    os.environ.get(
        "ROCQIPATH_NOTEBOOK_DEMO_DIR",
        str(PROJECT_ROOT / "notebook_demo_outputs"),
    )
)
synthetic_root = demo_root / "patch_extraction"
synthetic_he = synthetic_root / "reference" / "Sample_0001_he.tif"
synthetic_aligned = (
    synthetic_root
    / "aligned"
    / "CD8"
    / "Sample_0001_he"
    / "aligned_cd8.ome.tiff"
)

synthetic_he.parent.mkdir(parents=True, exist_ok=True)
synthetic_aligned.parent.mkdir(parents=True, exist_ok=True)
Image.fromarray(
    np.full((64, 64, 3), (150, 95, 125), dtype=np.uint8)
).save(synthetic_he, format="TIFF")
Image.fromarray(
    np.full((64, 64, 3), (135, 90, 65), dtype=np.uint8)
).save(synthetic_aligned, format="TIFF")

demo_cfg = PatchExtractionConfig(
    he_dir=str(synthetic_he.parent),
    aligned_dir=str(synthetic_root / "aligned"),
    output_dir=str(synthetic_root / "results"),
    biomarker_folders=["CD8"],
    reference_pattern=r"^(?P<sample_id>Sample_\d{4})_he\.tiff?$",
    reference_name="he",
    moving_name="cd8",
    patch_size=32,
    stride=32,
    tissue_threshold=0.50,
    target_magnification=20.0,
    reference_source_magnification=20.0,
    target_source_magnification=20.0,
)
demo_summary = run_patch_extraction(demo_cfg)
print(demo_summary)


In [ ]:
HE_DIR = DATA_ROOT / "reference"

# This must follow the resolver contract:
# <ALIGNED_FOR_PATCHES>/<biomarker>/<sample>_<reference_name>/*.ome.tif*
ALIGNED_FOR_PATCHES = DATA_ROOT / "aligned_for_patches"
OUTPUT_ROOT = RESULTS_ROOT

BIOMARKERS = ["CD8"]
TARGET_MAGNIFICATION = 20.0

# Physical objective of the exported aligned WSI. For alignment level 0,
# this is the H&E reference level-0 objective (for example 40.0 or 80.0).
# Leave None only when the aligned OME-TIFF reopens with objective metadata.
REFERENCE_SOURCE_MAGNIFICATION = None
ALIGNED_SOURCE_MAGNIFICATION = None

RUN_PATCH_EXTRACTION = False
RUN_RECONSTRUCTION = False


## Stage alignment outputs for patch discovery

The alignment pipeline writes
`results/alignment/<case>/<case>_aligned_moving.ome.tiff`, while the patch
resolver expects
`<aligned_dir>/<biomarker>/<sample>_<reference_name>/*.ome.tif*`.

The helper below creates the resolver layout with hard links when
possible and copies as a fallback. Originals are not renamed or deleted.
Run it once after alignment, or create the same structure yourself.


In [ ]:
import os
import shutil


def stage_alignment_outputs(
    alignment_root: Path,
    staged_root: Path,
    biomarker: str,
    reference_name: str = "he",
    *,
    copy_if_link_fails: bool = True,
) -> list[Path]:
    '''Stage standardized alignment cases for current patch discovery.'''
    marker_token = biomarker.lower()
    staged: list[Path] = []

    for case_dir in sorted(alignment_root.glob(f"*_{marker_token}")):
        if not case_dir.is_dir():
            continue
        sample_id = case_dir.name[: -(len(marker_token) + 1)]
        candidates = sorted(case_dir.glob("*.ome.tif*"))
        if not candidates:
            continue

        source = candidates[0]
        destination = (
            staged_root
            / biomarker
            / f"{sample_id}_{reference_name}"
            / f"aligned_{marker_token}.ome.tiff"
        )
        destination.parent.mkdir(parents=True, exist_ok=True)

        if destination.exists():
            staged.append(destination)
            continue
        try:
            os.link(source, destination)
        except OSError:
            if not copy_if_link_fails:
                raise
            shutil.copy2(source, destination)
        staged.append(destination)
    return staged


# Example (uncomment after alignment):
# staged = stage_alignment_outputs(
#     RESULTS_ROOT / "alignment",
#     ALIGNED_FOR_PATCHES,
#     biomarker="CD8",
# )
# print(*staged, sep="\n")


In [ ]:
patch_cfg = PatchExtractionConfig(
    he_dir=str(HE_DIR),
    aligned_dir=str(ALIGNED_FOR_PATCHES),
    output_dir=str(OUTPUT_ROOT),
    biomarker_folders=BIOMARKERS,
    reference_pattern=r"^(?P<sample_id>.+?)_he\.tiff?$",
    reference_name="he",
    moving_name="ihc",
    patch_size=512,
    stride=512,
    tissue_threshold=0.50,
    target_magnification=TARGET_MAGNIFICATION,
    reference_source_magnification=REFERENCE_SOURCE_MAGNIFICATION,
    target_source_magnification=ALIGNED_SOURCE_MAGNIFICATION,
    dimension_tolerance=0.01,
    max_workers=4,
)

print(patch_cfg.to_dict())


In [ ]:
if RUN_PATCH_EXTRACTION:
    patch_summary = run_patch_extraction(patch_cfg)
    print(f"Processed: {patch_summary['processed']}")
    print(f"Skipped  : {patch_summary['skipped']}")
    for case in patch_summary["cases"]:
        print(case)
else:
    patch_summary = None
    print("Set RUN_PATCH_EXTRACTION=True after staging aligned files.")


## Inspect the patch manifest

All channel images and the metadata file live in one shallow case folder:

```text
results/patch_extraction/Sample_0001_CD8/
├── Sample_0001_CD8_he_patch_000001.png
├── Sample_0001_CD8_ihc_patch_000001.png
└── Sample_0001_CD8_metadata.json
```

The manifest is the authoritative pairing/coordinate source.


In [ ]:
import json

case_manifests = sorted(
    (OUTPUT_ROOT / "patch_extraction").glob("*/*_metadata.json")
) if (OUTPUT_ROOT / "patch_extraction").is_dir() else []

print(f"Case manifests: {len(case_manifests)}")
if case_manifests:
    manifest_path = case_manifests[0]
    metadata = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(f"Case          : {metadata['case_id']}")
    print(f"Dimensions    : {metadata['dimensions']}")
    print(f"Patch size    : {metadata['patch_size']}")
    print(f"Stride        : {metadata['stride']}")
    print(f"Magnification : {metadata['target_magnification']}x")
    print(f"Saved patches : {len(metadata['patches'])}")
    print("\nFirst patch:")
    print(json.dumps(metadata["patches"][0], indent=2))


In [ ]:
from rocqipath.visualization import view_pairs

if case_manifests:
    case_dir = case_manifests[0].parent
    view_pairs(str(case_dir), num_to_show=min(5, len(metadata["patches"])))
else:
    print("Run patch extraction first.")


## Reconstruct model predictions

Reconstruction uses manifest coordinates. If `stride < patch_size`,
overlapping pixels are averaged; otherwise patches are pasted directly.
Missing patches are reported and left blank.

Put exactly one predicted image per six-digit patch ID in
`results/patch_extraction/<case_id>/predicted_ihc/`, then use
`mode="predicted_ihc"`. This avoids ambiguity in the current reconstruction
resolver when a flat case folder contains both H&E and IHC images with the
same patch ID.


In [ ]:
from rocqipath.extraction import ReversiblePatchExtractor

CASE_ID = "Sample_0001_CD8"
BIOMARKER = "CD8"

if RUN_RECONSTRUCTION:
    predicted_dir = (
        OUTPUT_ROOT
        / "patch_extraction"
        / CASE_ID
        / "predicted_ihc"
    )
    if not predicted_dir.is_dir():
        raise FileNotFoundError(
            f"Save one prediction per patch ID under: {predicted_dir}"
        )
    reconstructor = ReversiblePatchExtractor(
        {
            "he_root": str(HE_DIR),
            "aligned_root": str(ALIGNED_FOR_PATCHES),
            "output_dir": str(OUTPUT_ROOT),
            "biomarker_folders": BIOMARKERS,
            "patch_size": patch_cfg.patch_size,
            "stride": patch_cfg.stride,
            "target_magnification": TARGET_MAGNIFICATION,
        }
    )
    prediction_result = reconstructor.reconstruct_wsi(
        case_id=CASE_ID,
        biomarker=BIOMARKER,
        output_path=str(OUTPUT_ROOT),
        mode="predicted_ihc",
    )
    print("Predicted IHC:", prediction_result)
else:
    print(
        "Set RUN_RECONSTRUCTION=True after checking CASE_ID, manifest, "
        "and predicted_ihc files."
    )


## Practical choices

- `stride == patch_size`: non-overlapping dataset and simple reconstruction.
- `stride < patch_size`: more training samples, correlated overlap, and
  averaging during reconstruction.
- Lower `tissue_threshold` keeps edge patches; higher values emphasize dense
  tissue.
- `max_workers` parallelizes cases, not patches. Start with 2–4 workers and
  monitor storage throughput.
- Keep H&E and aligned IHC source fallbacks separate when scanner metadata
  differ.
